# 07 — CIFAR100 / Learned

The complete proposed procedure on ten tasks.

Select a **TensorFlow 2.20 / Keras 3** kernel, restart the kernel, then **Run All**.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "semantic_consolidation/config.py").is_file())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from notebooks.thesis.workflow import check_runtime
print(check_runtime())
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
from common.dataloader import get_datasets
from common.model import get_model
from common.train import train_model
from notebooks.thesis.workflow import load_run, attach_route, close_run, finish_run
from notebooks.thesis.presentation import describe_run, show_learning_results, show_diagnostics, show_saved_replay

CAMPAIGN = ROOT / "results/thesis_route_one/minimum_v4_tf220"

### 1. Select one stream

Notebook01 must have frozen this runtime and recipe. Selection and pairing are automatic.

In [ ]:
SHOW_DIAGNOSTICS = False  # Optional saved validation/replay views; never change the frozen recipe.
config, context = load_run(CAMPAIGN / "frozen_design.json", "cifar100", "learned",
                           repeat_index=None)
describe_run(config)

### 2. Load data and create the model

The common APIs own splitting, replay and class growth. The route attaches to this same model.

In [ ]:
project = config.common
trainset, valset = get_datasets(project)
bundle = get_model(project)
attach_route(context, bundle)

### 3. Train once

After an interruption, restart the kernel and **Run All** to resume the same stream. Work after the latest valid checkpoint is repeated.

In [ ]:
if context.get("training_started"):
    raise RuntimeError("Restart the kernel before training another stream.")
context["training_started"] = True
try:
    history = train_model(project, bundle, trainset, valset=valset)
except BaseException:
    close_run(context, release=True)
    raise
finally:
    close_run(context)

### 4. Save and read the results

Accuracy is percent; forgetting and backward transfer are signed percentage points. These are test outcomes from one complete stream.

In [ ]:
evaluations = finish_run(context, config, bundle, history, trainset, valset)
RUN, VIEW_DIR = show_learning_results(config, bundle, details=SHOW_DIAGNOSTICS)

### Optional: saved diagnostics

Enabled by SHOW_DIAGNOSTICS above. Phase changes use matched validation examples; replay grids are qualitative. These views are also available from notebook 10.

In [ ]:
if SHOW_DIAGNOSTICS:
    review = show_diagnostics(RUN, VIEW_DIR)
    show_saved_replay(config, bundle, RUN, VIEW_DIR)

Follow the next row in the saved execution checklist using a fresh kernel. This notebook selects one next unfinished repeat; it does not launch a sweep. After all **24 streams**, run **10_Collect_Thesis_Results.ipynb**. Keep negative or uncertain results and do not change the frozen design.